# MathGlyph Pages Generation Examples

This notebook works in Google Colab from a fresh runtime. It installs `mathglyph-pages` when the repo source tree is not available, downloads a curated set of real MathWriting InkML samples from this repository, and then shows how to control page type, formula density, scan/photo distortions, and MathWriting split selection. The selected samples are full expressions rather than standalone symbols. For real generation, point `MATHWRITING_ROOT` to a mounted MathWriting 2024 dataset root.


In [ ]:

from pathlib import Path
import importlib.util
import json
import subprocess
import sys
import tempfile
import urllib.request

from IPython.display import display

cwd = Path.cwd()
repo_root = None
for candidate in (cwd, cwd.parent):
    if (candidate / "src" / "mathglyph_pages").exists():
        repo_root = candidate
        sys.path.insert(0, str(candidate / "src"))
        break

if repo_root is None and importlib.util.find_spec("mathglyph_pages") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "git+https://github.com/reirei-00/mathglyph_pages.git",
    ])

from PIL import Image, ImageDraw
from mathglyph_pages import MathPageConfig, generate_pages
from mathglyph_pages.inkml import parse_inkml

SAMPLE_FILES = [
    "valid/009f7b727b467490.inkml",
    "valid/00235ec3dee919dd.inkml",
    "valid/011bc2c4d3e15714.inkml",
    "valid/017a79eea7fa5f63.inkml",
    "valid/00b2ce18a5aa30a6.inkml",
    "valid/00253e181f0200ab.inkml",
    "valid/006fb32b84a99137.inkml",
    "synthetic/0000d71cda4329a6.inkml",
    "synthetic/000b80a1a1f5a854.inkml",
    "train/0578f6b97d18358a.inkml",
]


def download_sample_mathwriting(root: Path) -> Path:
    base = "https://raw.githubusercontent.com/reirei-00/mathglyph_pages/main/examples/mathwriting_samples"
    for rel_path in SAMPLE_FILES:
        dst = root / rel_path
        dst.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{base}/{rel_path}", dst)
    return root


def discover_splits(root: Path) -> tuple[str, ...]:
    splits = tuple(sorted(path.name for path in root.iterdir() if path.is_dir() and list(path.glob("*.inkml"))))
    if not splits:
        raise RuntimeError(f"No InkML split directories found under {root}")
    return splits


WORKDIR_HANDLE = tempfile.TemporaryDirectory()
WORKDIR = Path(WORKDIR_HANDLE.name)

if repo_root is not None and (repo_root / "examples" / "mathwriting_samples").exists():
    MATHWRITING_ROOT = repo_root / "examples" / "mathwriting_samples"
else:
    MATHWRITING_ROOT = download_sample_mathwriting(WORKDIR / "mathwriting_samples")

MATHWRITING_SPLITS = discover_splits(MATHWRITING_ROOT)

print("MathWriting sample root:", MATHWRITING_ROOT)
print("Splits:", MATHWRITING_SPLITS)
print("Temporary output dir:", WORKDIR)
print("Sample labels:")
for path in sorted(MATHWRITING_ROOT.glob("*/*.inkml"))[:10]:
    formula = parse_inkml(path)
    print("-", path.parent.name, formula.sample_id, formula.label)


In [ ]:

REGION_COLORS = {
    "formula": "#e11d48",
    "handwritten": "#2563eb",
    "printed": "#111827",
    "annotation": "#16a34a",
    "table": "#f97316",
    "image": "#7c3aed",
    "graph": "#7c3aed",
}


def load_metadata(result):
    return [json.loads(line) for line in result.metadata_path.read_text(encoding="utf-8").splitlines() if line.strip()]


def display_region_overlay(
    result,
    page_index: int = 0,
    max_width: int = 950,
    *,
    show_formula_indexes: bool = False,
    show_label_chips: bool = True,
):
    rows = load_metadata(result)
    row = rows[page_index]
    image = Image.open(result.out_dir / row["file_name"]).convert("RGB")
    overlay = image.copy()
    draw = ImageDraw.Draw(overlay)
    for region in row["regions"]:
        role = region.get("role")
        if role == "formula_index" and not show_formula_indexes:
            continue
        rtype = region.get("subtype") if region.get("subtype") == "graph" else region.get("type", "unknown")
        color = REGION_COLORS.get(str(rtype), "#0f766e")
        x0, y0, x1, y1 = [int(v) for v in region["bbox"]]
        width = 2 if role == "formula_index" else 5
        draw.rectangle((x0, y0, x1, y1), outline=color, width=width)
        box_w = x1 - x0
        box_h = y1 - y0
        if not show_label_chips or box_w < 58 or box_h < 22:
            continue
        label = "index" if role == "formula_index" else str(rtype)
        text_y = max(0, y0 - 18)
        text_bbox = draw.textbbox((x0, text_y), label)
        draw.rectangle((text_bbox[0] - 2, text_bbox[1] - 1, text_bbox[2] + 2, text_bbox[3] + 1), fill=color)
        draw.text((x0, text_y), label, fill="white")
    if overlay.width > max_width:
        scale = max_width / overlay.width
        overlay = overlay.resize((max_width, round(overlay.height * scale)))
    display(overlay)


def run_example(name: str, **kwargs):
    kwargs.setdefault("include_annotations", False)
    config = MathPageConfig(
        mathwriting_root=MATHWRITING_ROOT,
        out_dir=WORKDIR / name,
        splits=MATHWRITING_SPLITS,
        max_scan_per_split=None,
        **kwargs,
    )
    result = generate_pages(config)
    summary = json.loads(result.summary_path.read_text(encoding="utf-8"))
    print(name, "pages=", summary["num_pages"], "regions=", summary["num_regions"])
    print("profiles:", summary["page_profiles"])
    print("styles:", summary["visual_styles"])
    print("region counts:", summary["region_type_counts"])
    if result.contact_sheet_path:
        display(Image.open(result.contact_sheet_path))
    print("Highlighted regions on first page, annotations and formula indexes hidden for cleaner examples:")
    display_region_overlay(result, page_index=0)
    return result, summary


## Clean sparse pages

Use `profile="formula_sparse"` for fewer formulas and `visual_style="clean"` for minimal augmentation.

In [ ]:
clean_sparse, clean_sparse_summary = run_example(
    "clean_sparse",
    num_pages=2,
    seed=101,
    profile="formula_sparse",
    visual_style="clean",
    formulas_per_page_min=4,
    formulas_per_page_max=5,
)

## Dense pages

Density is controlled by the page profile and the explicit formula-count bounds.

In [ ]:
dense, dense_summary = run_example(
    "dense",
    num_pages=2,
    seed=202,
    profile="formula_dense",
    visual_style="noisy_scan",
    formulas_per_page_min=12,
    formulas_per_page_max=14,
    formula_target_height_min=62,
    formula_target_height_max=88,
)

## Distortion controls

Choose a visual preset or override individual distortion fields directly.

In [ ]:
distorted, distorted_summary = run_example(
    "distorted",
    num_pages=2,
    seed=303,
    profile="formula_scattered",
    visual_style="clean",
    rotation_max_degrees=3.0,
    perspective_strength=0.05,
    yellow_strength=0.18,
    noise_strength=11,
    blur_radius=0.15,
    edge_shadow=True,
)

## Tables and matrix-heavy pages

`formula_matrix_table` requests matrix/array labels from MathWriting for part of the page.

In [ ]:
matrix_table, matrix_table_summary = run_example(
    "matrix_table",
    num_pages=1,
    seed=404,
    profile="formula_matrix_table",
    visual_style="aged_scan",
    formulas_per_page_min=6,
    formulas_per_page_max=8,
)

## Inspect metadata

Every formula region records which MathWriting split/file/sample drove it.

In [ ]:
rows = [json.loads(line) for line in matrix_table.metadata_path.read_text(encoding="utf-8").splitlines()]
formula_regions = [region for region in rows[0]["regions"] if region["type"] == "formula"]
formula_regions[:3]